# BioTek Cytation 5

```{device-card} biotek-cytation-5
```

| Property | Driver behavior |
| --- | --- |
| Controller | FTDI USB or a serial port; 9600/38400 baud detection, 8N2, hardware flow control |
| Camera | Aravis / USB3 Vision, without Gen5, FlyCapture or Spinnaker |
| Optics | Six objective slots, installed fluorescence cubes, brightfield and phase contrast |
| Focus | Manual focus, bounded image autofocus, calibrated laser autofocus |
| Acquisition | Mono8 and unpacked Mono10/12/16; exposure, gain, overlapping montages |
| Other functions | Plate-reader assays, incubation and tray control |

Hardware development used a CYT5MPV (2017), firmware 1.12, Grasshopper3 GS3-U3-14S5M,
4×/10×/20× phase objectives, 4× brightfield, laser cube, DAPI, RFP and Texas Red.
The plate was a Corning 3603, represented by `cor_96_wellplate_360uL_Fb`.
BioSpa is a separate device and is not controlled by this driver.

Microscope heights and offsets are in **mm**, camera exposure in **ms**, gain in **dB**,
and wells use **one-based** row and column numbers. Plate-reader assay focus is a separate
optical setting from microscope focus.

## Connections and installation

Connect both USB cables: the controller and the camera enumerate separately. Connect the camera
to a USB3 port. Power on the reader and let initialization finish. Close any other application
holding the controller or camera. Gen5 does not need to be installed.

### Windows

Keep the controller on its FTDI virtual COM-port driver. For the Grasshopper camera, use
[Zadig](https://zadig.akeo.ie/) to install **WinUSB on camera interface 0** (VID `1E10`, PID `3300`
on the tested camera). Do not replace the controller's driver. Replacing a FlyCapture camera
driver changes which applications can open that camera; save its old driver if you need to switch back.

Install [MSYS2](https://www.msys2.org/) and run these commands in its **UCRT64** shell:

```sh
pacman -S --needed mingw-w64-ucrt-x86_64-python \
  mingw-w64-ucrt-x86_64-python-pip mingw-w64-ucrt-x86_64-python-gobject \
  mingw-w64-ucrt-x86_64-python-numpy mingw-w64-ucrt-x86_64-aravis
python -m venv --system-site-packages .venv
source .venv/bin/activate
python -m pip install -e '.[serial]'
```

Run the last command from this PyLabRobot checkout. Use the same Python environment for the
notebook kernel. The system-site-packages option exposes MSYS2's matching PyGObject and NumPy builds.
Do not mix those native extensions with python.org's Windows interpreter.
See [MSYS2 Python](https://www.msys2.org/docs/python/) and its
[Aravis package](https://packages.msys2.org/packages/mingw-w64-ucrt-x86_64-aravis).

### Linux and macOS

Install Aravis 0.8 and Python bindings for its GObject interface, plus NumPy and PyLabRobot's
serial or FTDI dependencies. On Debian/Ubuntu, the system packages include `libaravis-0.8-0`,
`gir1.2-aravis-0.8`, `python3-gi` and `python3-numpy`; a virtual environment with
`--system-site-packages` can use those bindings. On macOS, install `aravis` and `pygobject3`
with Homebrew and use the Python interpreter matching those bindings.

USB permissions must allow the Python process to open the camera. Windows is the hardware-tested
installation in this guide; the driver contains no Windows-specific hardware calls.

## Import the driver

Import the imaging types and the plate definition.

In [ ]:
import asyncio
import numpy as np

from pylabrobot.agilent.biotek.cytation import Cytation5, CytationImagingConfig
from pylabrobot.agilent.biotek.cytation.microscope.aravis_camera import AravisCamera
from pylabrobot.agilent.biotek.cytation.microscope.models import AutoFocus, ImagingMode, Objective
from pylabrobot.resources import cor_96_wellplate_360uL_Fb

## Discover the camera

Discovery does not start acquisition. When selecting a camera explicitly, use its discovery
identifier; USB discovery and the manufacturer's label can express the serial number differently.

In [ ]:
AravisCamera.enumerate_cameras()

## Connect

Set the controller's actual port. Alternatively omit `port` and pass the FTDI adapter's `device_id`.
`use_cam=False` enables controller-only operation. Configuration overrides are optional; normal setup
reads the installed objectives, filters, focus resolution and motor-field width from the instrument.

In [ ]:
reader = Cytation5(name="cytation", port="COM9")
await reader.setup()
m = reader.microscope

## Inspect installed optics

Empty turret slots remain `None`. The laser cube is exposed through `m.laser` rather than as a
fluorescence imaging mode.

In [ ]:
m.objective_configurations, m.filters, m.laser.available

## Extend the tray

Keep the tray's travel clear. This also turns off illumination and invalidates any laser reference
for the previously loaded plate.

In [ ]:
await reader.open()

## Load the plate

Place the plate in the indicated orientation. Set `with_lid` to match the physical plate.
The definition includes the well spacing, bottom geometry and carrier height.

In [ ]:
plate = cor_96_wellplate_360uL_Fb("sample_plate", with_lid=True)

## Retract the tray

Pass the plate so the controller receives its geometry before closing. If geometry is changed
while closed and the firmware returns `0x5B00`, `set_plate()` automatically extends the tray,
retries the geometry, then retracts it. Other errors are raised.

In [ ]:
await reader.close(plate=plate)

## Select an objective and well

This example uses the installed 4× phase objective in brightfield mode. Phase contrast uses
`ImagingMode.PHASE_CONTRAST`, which selects the annulus configured for that objective.

In [ ]:
objective = Objective.O_4X_PL_FL_Phase
await m.set_objective(objective)
await m.set_imaging_mode(ImagingMode.BRIGHTFIELD, led_intensity=1)
await m.select(row=1, column=2)

## Position and focus

Offsets are relative to the well center and are quantized to the stage's 10.16 µm motor steps.
The factory focus reference is available through `get_focus_reference()`, but is not a measurement
of your sample plane. Choose focus using your plate geometry and an image; do not treat the factory
reference as the sample height.

A focus near 3.28 mm resolved the test plate bottom with the 4× phase objective. This is an example
for that configuration, not a universal plate-height calibration.

In [ ]:
await m.set_position(x=0, y=0)
await m.set_focus(3.28)

## Capture a frame

Select Mono8 before starting acquisition. A capture handles acquisition and switches off illumination
when it finishes, including on errors. Returned images are NumPy arrays.

In [ ]:
await m.camera.set_pixel_format()
result = await m.capture(
  row=1, column=2, mode=ImagingMode.BRIGHTFIELD, objective=objective,
  exposure_time=150, focal_height=3.28, gain=0, plate=plate,
)
image = result.images[0]

## Automatic exposure

For a capture, `"machine-auto"` performs a bounded software search and retains a fixed exposure.
It targets the 99th-percentile pixel level between 60% and 80% of the format's range.
`AutoExposure` settings can supply a custom evaluator and bounds for a specific sample.
A dark fluorescence sample may have no acceptable exposure within the camera's limits; that raises
an error instead of claiming a successful adjustment.

In [ ]:
result = await m.capture(
  row=1, column=2, mode=ImagingMode.BRIGHTFIELD, objective=objective,
  exposure_time="machine-auto", focal_height=3.28, gain=0, plate=plate,
)
result.exposure_time

## Image autofocus

Provide a finite search interval that is safe for the selected objective and plate. The example
metric measures high-frequency contrast. An empty or uniformly coloured well may have no usable
structure; use visible cells, surface detail or the well edge. This example uses an offset near
the test plate's well edge. Focus searches reject a flat metric or a best point at the search boundary.
Exposure and gain must be fixed during the search.

In [ ]:
def focus_score(image):
  values = image[::2, ::2].astype(float)
  return float(np.mean(np.diff(values, axis=0) ** 2) + np.mean(np.diff(values, axis=1) ** 2))

focused = await m.capture(
  row=1, column=2, mode=ImagingMode.BRIGHTFIELD, objective=objective,
  exposure_time=350, focal_height=AutoFocus(focus_score, timeout=180, low=3.0, high=3.5),
  gain=0, plate=plate, center_position=(2.5, 0),
)
focused.focal_height

## Laser autofocus

Laser autofocus tracks a chosen sample plane through the plate-bottom reflection. Calibrate with
an unobstructed bottom reflection after finding focus by imaging. The reference is specific to the
objective, imaging mode and plate type. Loading another plate invalidates it.

The calibration scan moves by `span` on either side of the supplied focus. It must stay inside the
objective's safe travel. Laser exposure is separate from sample-imaging exposure. Saturated, missing,
or inconsistent reflections raise an error. Laser current is checked against the internal monitor
before enabling the diode and is constrained below the cube's 30 mA absolute maximum.

This example calibrates the tested 4× phase objective at the flat bottom of an empty control well.

Calibration temporarily uses Mono16 for subpixel reflection measurement, then restores the previous
pixel format, exposure, gain and optical path. Starting settings tested on the Corning 3603 plate:

| Objective | Calibration span (±mm) | Laser exposure (ms) | Current (mA) |
| --- | --- | --- | --- |
| 4× phase | 0.2 | 5 | 12 |
| 10× phase | 0.05 | 20 | 12 |
| 20× phase | 0.05 | 50 | 12 |
| 4× brightfield | 0.5 | 10 | 12 |

These are starting settings, not universal calibrations. The 4× brightfield objective produces a
smaller reflection displacement and lower focus sensitivity. Calibration rejects weak, saturated
or nonlinear reflections. Recalibrate after changing plates or optical configuration.

In [ ]:
await m.set_objective(Objective.O_4X_PL_FL_Phase)
await m.set_imaging_mode(ImagingMode.BRIGHTFIELD, led_intensity=1)
await m.select(1, 3)
calibration = await m.laser.calibrate(focal_position=3.28, span=0.2, exposure=5, current=12)

Correct focus within a bounded interval. The routine restores the sample-imaging optical path
and camera settings, and leaves illumination off. `set_focus("machine-auto")` uses this calibration.

`tolerance` is the convergence threshold calculated from the reflection, not a guaranteed absolute
focus accuracy. Reflection contrast, plate geometry and objective magnification affect repeatability.
Check image sharpness for your sample; use image autofocus to refine a laser-referenced position
when the depth of field is narrower than the observed variation.

In [ ]:
focus = await m.laser.autofocus(start=3.4, max_correction=0.5, tolerance=0.005)
focus

## Overlapping montages

Coverage is `(rows, columns)`, or `"full"` for a grid covering the well. Overlap is a fraction in
`[0, 1)`. Images are returned in row-major order with actual quantized stage offsets in
`tile_positions` and the grid dimensions in `tile_grid`; stitching is left to image-analysis software.

Field of view uses the camera's rectangular sensor dimensions. The tested Grasshopper has
6.45 µm sensor pixels, giving a nominal 2.2317 × 1.67055 mm field at 4×. A 0.2032 mm stage displacement
measured approximately 127 image pixels, or 1.6 µm/pixel, on the test instrument.
For quantitative calibration or another camera, set `camera_pixel_size` (sensor pitch in mm) and
`optical_relay_magnification` in `CytationImagingConfig`.
The sensor pitch comes from the [manufacturer's camera specification](https://www.eureca.de/files/pdf/optoelectronics/flir/GS3-U3-Technical-Reference.pdf).

In [ ]:
montage = await m.capture(
  row=1, column=2, mode=ImagingMode.BRIGHTFIELD, objective=objective,
  exposure_time=150, focal_height=3.28, gain=0, plate=plate,
  coverage=(2, 2), overlap=0.2,
)
montage.tile_grid, montage.tile_positions

## Fluorescence channels

Use only installed cubes. Exposure and LED intensity need to be chosen for the fluorophore.
The food-colour test plate verifies acquisition paths but is not a fluorescence sensitivity standard.

In [ ]:
channels = {}
for mode in (ImagingMode.DAPI, ImagingMode.RFP, ImagingMode.TEXAS_RED):
  channels[mode.name] = await m.capture(
    row=1, column=2, mode=mode, objective=objective,
    exposure_time=100, focal_height=3.28, gain=0, plate=plate, led_intensity=1,
  )

## Colour brightfield

Three sequential red, green and blue illuminations produce an `(height, width, 3)` RGB array.
Exposure and gain are shared across channels. Automatic exposure evaluates the RGB image;
use fixed exposure for comparable colours across wells.

In [ ]:
colour = await m.capture(
  row=1, column=2, mode=ImagingMode.COLOR_BRIGHTFIELD, objective=objective,
  exposure_time="machine-auto", focal_height=3.28, gain=0, plate=plate,
  led_intensity=1,
)
colour.images[0].shape

## Higher intensity depth

Use unpacked Mono16 to retain the camera's full intensity precision. The Grasshopper3 sensor
provides 12-bit measurements aligned in a 16-bit container; arrays use native NumPy `uint16`
with the camera byte order handled by the driver. Change format while acquisition is stopped.

In [ ]:
await m.camera.set_pixel_format(0x01100007)  # GenICam Mono16
high_depth = await m.capture(
  row=1, column=2, mode=ImagingMode.BRIGHTFIELD, objective=objective,
  exposure_time=150, focal_height=3.28, gain=0, plate=plate,
)
high_depth.images[0].dtype

Restore Mono8 for the following examples.

In [ ]:
await m.camera.set_pixel_format()

## Z stacks and time series

Compose ordinary captures into a stack. Specify a bounded set of heights appropriate for the sample.
For a time series, repeat captures at the desired interval using `asyncio.sleep`; illumination stays
off between captures. Images can be saved losslessly with NumPy or a TIFF library.

In [ ]:
stack = []
for height in (3.26, 3.28, 3.30):
  frame = await m.capture(
    row=1, column=2, mode=ImagingMode.BRIGHTFIELD, objective=objective,
    exposure_time=150, focal_height=height, gain=0, plate=plate,
  )
  stack.append(frame.images[0])
np.savez_compressed("cytation-stack.npz", images=np.stack(stack), heights=[3.26, 3.28, 3.30])

## Incubation

Cytation 5 supports heating up to 65 °C, with regulation specified from ambient + 4 °C.
Setting a target acknowledges the command; it does not wait until the plate reaches that temperature.
A lower target permits passive cooling. There is no active refrigeration.

The tested instrument returned `0x151F` when enabling heating. Gen5 independently identified the
same error as an incubator thermistor fault. Temperature queries and disabling heat were verified;
heating regulation needs verification after that instrument fault is repaired. Do not interpret a
successful setpoint command as a measured plate temperature. The optional `gradient` argument
sets the top/bottom temperature difference; firmware determines its range and resolution.

In [ ]:
await reader.set_temperature(37)
await reader.get_temperature()

Disable incubation explicitly when it is no longer needed. `stop()` releases connections while
preserving an incubation target, so closing a program does not silently end incubation.

In [ ]:
await reader.stop_heating_or_cooling()

## Plate-reader assays

The same controller supports plate-reader measurements. Switching to an assay turns off microscope
illumination and invalidates microscope positioning. Pass actual plate wells to select a subset.

A subsequent microscope `capture()` closes the tray with the selected plate geometry and restores
imaging settings. For low-level manual imaging, call `close(plate=plate)` after an assay or shaking.
`home()` moves the imager to its home position and extends the tray.

In [ ]:
absorbance = await reader.read_absorbance(
  plate=plate, wells=[plate.get_item("A1"), plate.get_item("A2"), plate.get_item("A3")],
  wavelength=450,
)
absorbance[0].data[0][:3]

Fluorescence and luminescence use the plate-reader optical path, separate from the microscope.
Choose wavelengths and focal height for the assay. Results contain the plate-shaped data array,
measurement temperature and timestamp; unselected wells contain `None`.

In [ ]:
fluorescence = await reader.read_fluorescence(
  plate=plate, wells=[plate.get_item("A3")],
  excitation_wavelength=485, emission_wavelength=520, focal_height=6,
)
luminescence = await reader.read_luminescence(
  plate=plate, wells=[plate.get_item("A3")], focal_height=6, integration_time=0.2,
)

## Shaking

Shaking runs until stopped. Frequency is the instrument speed setting: 6 means 360 cycles/min,
5 = 410, 4 = 493, 3 = 567, 2 = 731 and 1 = 1096. Stop shaking before imaging or reading.

In [ ]:
await reader.shake(reader.ShakeType.ORBITAL, frequency=6)
try:
  await asyncio.sleep(2)
finally:
  await reader.stop_shaking()

## Shutdown

Stop any ongoing experiment, then release the camera and controller. This turns off microscope
illumination. After a protocol timeout, verify that the instrument is idle before reconnecting;
a failed command is not proof that motion never started.

In [ ]:
await reader.stop()

## Hardware validation and limits

The Windows implementation was exercised on the configuration listed above with the Gen5
installation directory temporarily unavailable. The sequence included laser reference correction,
Mono8/Mono16 capture, absorbance followed by microscopy, orbital shaking followed by microscopy,
and automatic-exposure RGB capture. Mono8 and Mono16 images of the same field had a correlation
of 0.993 after correcting the Grasshopper camera's 16-bit byte order.

Separate runs exercised all four objectives, phase annuli, DAPI/RFP/Texas Red illumination,
image autofocus, exposure search, overlapping tiles, fluorescence and luminescence plate reads.
Food colour and an empty well establish command and image acquisition behavior; they do not
establish fluorescence sensitivity or assay accuracy. The incubator thermistor fault described
above prevented heating regulation validation. No BioSpa connection is required.